# AI기반 전통시장 여행 플래너 - Gradio 프로토타입 (Colab)

이 노트북은 아래 순서로 실행하면 됩니다.

1. **셀 1** 실행 → gradio 설치
2. 왼쪽 파일 탭에 **`전통시장_상가매칭_광주전남_FULL.csv`**를 드래그해서 업로드
   (이 파일 하나만 있으면 노트북이 스스로 MARKET·MARKET_STALL 테이블을 채웁니다)
3. **셀 2** 실행 → DB 스키마 생성 + CSV 자동 적재 + 화면 로직 정의
4. **셀 3** 실행 → `https://xxxx.gradio.live` 공유 링크 생성, 클릭해서 접속

> 이미 만들어둔 `travel_planner.db` 파일이 있다면, 그 파일을 대신 업로드해도 됩니다.
> 이 경우 MARKET_STALL에 데이터가 이미 있는 걸 감지해서 CSV 적재를 자동으로 건너뜁니다.

## 이번 업데이트 (5.기능요구사항 반영)
- 회원가입/로그인 방식: 카카오 / 네이버 / 구글 / 자체(이메일) 4종 (FR-001~006)
  - ⚠️ 프로토타입 한계: Colab에서는 실제 OAuth 리다이렉트가 어려워, 카카오/네이버/구글은 이메일 입력만으로
    "그 방식으로 가입/로그인했다"고 시뮬레이션합니다. 실서비스 배포 시 각 사 SDK 연동으로 교체가 필요해요.
- 온보딩 "선호 여행 스타일" 다중선택 체크박스 7개 (FR-007) — PRD에 구체 항목이 없어 자연/힐링, 맛집/미식,
  전통시장, 문화/역사, 액티비티/체험, 카페/디저트, 사진/인생샷 으로 정의했습니다. 원하시면 자유롭게 바꾸세요.
- 자체(이메일) 가입 시 비밀번호 재확인 필드 추가, 8자 미만/불일치 시 에러 처리 (FR-005)
- 로그인 화면에 로그아웃 버튼 추가 (FR-006)


## 1. 패키지 설치

In [ ]:
!pip install -q gradio

## 2. 앱 코드 (DB 스키마 + CSV 자동 적재 + 화면 로직)
- ERD에서 확정한 23개 테이블 CREATE문이 `SCHEMA_SQL`에 그대로 들어있습니다.
- `전통시장_상가매칭_광주전남_FULL.csv`를 읽어서 `MARKET`(시장 단위로 그룹핑) + `MARKET_STALL`(점포 하나하나)에 자동으로 채워 넣습니다.
- 이미 데이터가 들어있으면(재실행 등) 중복으로 쌓이지 않고 자동으로 건너뜁니다.

In [ ]:
# ============================================================
# AI기반 지역관광 맞춤 여행 플래너 - Colab용 Gradio 프로토타입
# ------------------------------------------------------------
# 사용법 (Colab):
#   1) !pip install -q gradio
#   2) travel_planner.db 파일을 Colab에 업로드 (좌측 파일 탭에 드래그)
#      또는 Google Drive 마운트 후 경로 수정
#   3) 이 셀 전체 실행 -> 마지막 줄이 공유 링크(gradio.live)를 출력
# ------------------------------------------------------------
# 이 프로토타입이 다루는 화면 (드로우아이오 흐름도 기준):
#   - 회원가입 / 로그인 (일반 사용자)
#   - 지역 검색 (전통시장 둘러보기)
#   - 시장 상세 (내부 점포 목록 + 업종 필터)
#   - 찜하기 (즐겨찾기)
#   - 마이페이지 (내 정보 + 찜 목록)
# 현재 DB에는 MARKET(121) / MARKET_STALL(10,264)만 데이터가 차있고
# USER 등 나머지 테이블은 비어있어서, 회원가입을 하면 그 자리에서
# USER 테이블에 실제로 INSERT 되는 구조입니다.
# ============================================================

import sqlite3
import uuid
import hashlib
from datetime import datetime

import gradio as gr

DB_PATH = "travel_planner.db"  # Colab에 업로드한 파일명과 동일해야 함 (없으면 이 코드가 새로 만듦)


# ============================================================
# ERD 확정본 SQL CREATE문 (schema.sql 원본 그대로, IF NOT EXISTS만 추가)
# - 이미 travel_planner.db를 업로드해서 데이터가 들어있다면: 아무것도 덮어쓰지 않고 그대로 사용됩니다.
# - DB 파일이 없거나 새 빈 DB라면: 아래 23개 테이블 + 인덱스가 이 스키마 그대로 생성됩니다.
# ============================================================
SCHEMA_SQL = """
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS USER (
  user_id TEXT PRIMARY KEY,
  name TEXT NOT NULL,
  email TEXT,
  birthdate TEXT,
  auth_provider TEXT CHECK(auth_provider IN ('kakao','naver','google','pass','email')),
  auth_provider_id TEXT,
  identity_verified INTEGER DEFAULT 0,
  status TEXT DEFAULT 'active',
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS ADMIN (
  admin_id TEXT PRIMARY KEY,
  name TEXT NOT NULL,
  email TEXT,
  role TEXT,
  invited_by TEXT REFERENCES ADMIN(admin_id),
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS MARKET (
  market_id TEXT PRIMARY KEY,
  market_name TEXT NOT NULL,
  market_type TEXT,
  open_cycle TEXT,
  address_road TEXT,
  address_jibun TEXT,
  lat REAL,
  lon REAL,
  store_count INTEGER,
  item_categories TEXT,
  phone TEXT,
  data_ref_date TEXT,
  managed_by TEXT REFERENCES ADMIN(admin_id)
);

CREATE TABLE IF NOT EXISTS MERCHANT (
  merchant_id TEXT PRIMARY KEY,
  name TEXT NOT NULL,
  business_number TEXT,
  phone TEXT,
  email TEXT,
  auth_provider TEXT,
  market_id TEXT REFERENCES MARKET(market_id),
  approval_status TEXT DEFAULT 'pending' CHECK(approval_status IN ('pending','approved','rejected')),
  approved_by TEXT REFERENCES ADMIN(admin_id),
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS CONVERSATION (
  conversation_id TEXT PRIMARY KEY,
  user_id TEXT NOT NULL REFERENCES USER(user_id),
  title TEXT,
  is_deleted INTEGER DEFAULT 0,
  created_at TEXT DEFAULT CURRENT_TIMESTAMP,
  updated_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS MESSAGE (
  message_id TEXT PRIMARY KEY,
  conversation_id TEXT NOT NULL REFERENCES CONVERSATION(conversation_id),
  sender TEXT CHECK(sender IN ('user','ai')),
  content TEXT,
  extracted_slots TEXT,
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS PLACE (
  place_id TEXT PRIMARY KEY,
  name TEXT NOT NULL,
  category TEXT,
  address TEXT,
  lat REAL,
  lon REAL,
  description TEXT,
  source TEXT
);

CREATE TABLE IF NOT EXISTS ITINERARY (
  itinerary_id TEXT PRIMARY KEY,
  user_id TEXT NOT NULL REFERENCES USER(user_id),
  conversation_id TEXT REFERENCES CONVERSATION(conversation_id),
  title TEXT,
  travel_type TEXT,
  start_date TEXT,
  end_date TEXT,
  version INTEGER DEFAULT 1,
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS ITINERARY_PLACE (
  itinerary_place_id TEXT PRIMARY KEY,
  itinerary_id TEXT NOT NULL REFERENCES ITINERARY(itinerary_id),
  place_id TEXT REFERENCES PLACE(place_id),
  market_id TEXT REFERENCES MARKET(market_id),
  day_number INTEGER,
  visit_order INTEGER,
  visit_time TEXT
);

CREATE TABLE IF NOT EXISTS ITINERARY_HISTORY (
  history_id TEXT PRIMARY KEY,
  itinerary_id TEXT NOT NULL REFERENCES ITINERARY(itinerary_id),
  change_type TEXT CHECK(change_type IN ('추가','삭제','교체','재정렬')),
  snapshot TEXT,
  changed_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS MARKET_STALL (
  stall_id TEXT PRIMARY KEY,
  market_id TEXT NOT NULL REFERENCES MARKET(market_id),
  merchant_id TEXT REFERENCES MERCHANT(merchant_id),
  stall_number TEXT,
  pos_x REAL,
  pos_y REAL,
  stall_type TEXT,
  -- 아래는 공공데이터(상가업소 상권정보) 매칭 원본을 보존하기 위한 컬럼
  -- 상인이 실제 회원가입하면 merchant_id로 연결하고, 이 컬럼들은 참고용으로 남김
  biz_name TEXT,
  category_large TEXT,
  category_mid TEXT,
  category_small TEXT,
  road_address TEXT,
  distance_m REAL,
  source_biz_id TEXT,
  data_source TEXT DEFAULT '소진공_상가업소_202603'
);

CREATE TABLE IF NOT EXISTS MARKET_FLOOR_MAP (
  floor_map_id TEXT PRIMARY KEY,
  market_id TEXT NOT NULL REFERENCES MARKET(market_id),
  map_image_url TEXT,
  map_type TEXT,
  updated_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS QR_CODE (
  qr_id TEXT PRIMARY KEY,
  stall_id TEXT NOT NULL REFERENCES MARKET_STALL(stall_id),
  qr_image_url TEXT,
  issued_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS QR_VISIT_LOG (
  visit_id TEXT PRIMARY KEY,
  user_id TEXT NOT NULL REFERENCES USER(user_id),
  qr_id TEXT NOT NULL REFERENCES QR_CODE(qr_id),
  visited_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS REVIEW (
  review_id TEXT PRIMARY KEY,
  user_id TEXT NOT NULL REFERENCES USER(user_id),
  target_type TEXT CHECK(target_type IN ('place','market','stall')),
  target_id TEXT,
  rating INTEGER CHECK(rating BETWEEN 1 AND 5),
  content TEXT,
  qr_verified INTEGER DEFAULT 0,
  source_visit_id TEXT REFERENCES QR_VISIT_LOG(visit_id),
  is_reported INTEGER DEFAULT 0,
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS FAVORITE (
  favorite_id TEXT PRIMARY KEY,
  user_id TEXT NOT NULL REFERENCES USER(user_id),
  target_type TEXT,
  target_id TEXT,
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS STAMP_TOUR (
  stamp_id TEXT PRIMARY KEY,
  user_id TEXT NOT NULL REFERENCES USER(user_id),
  market_id TEXT NOT NULL REFERENCES MARKET(market_id),
  stamps_collected TEXT,
  completed_at TEXT
);

CREATE TABLE IF NOT EXISTS NOTICE (
  notice_id TEXT PRIMARY KEY,
  author_type TEXT CHECK(author_type IN ('merchant','admin')),
  author_id TEXT,
  market_id TEXT REFERENCES MARKET(market_id),
  title TEXT,
  content TEXT,
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS COUPON (
  coupon_id TEXT PRIMARY KEY,
  issuer_type TEXT CHECK(issuer_type IN ('merchant','admin')),
  issuer_id TEXT,
  stall_id TEXT REFERENCES MARKET_STALL(stall_id),
  market_id TEXT REFERENCES MARKET(market_id),
  discount_info TEXT,
  valid_from TEXT,
  valid_to TEXT
);

CREATE TABLE IF NOT EXISTS RESERVATION (
  reservation_id TEXT PRIMARY KEY,
  user_id TEXT NOT NULL REFERENCES USER(user_id),
  stall_id TEXT NOT NULL REFERENCES MARKET_STALL(stall_id),
  item_info TEXT,
  status TEXT DEFAULT '대기' CHECK(status IN ('대기','확인','완료','취소')),
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS IMPROVEMENT (
  improvement_id TEXT PRIMARY KEY,
  market_id TEXT NOT NULL REFERENCES MARKET(market_id),
  stall_id TEXT REFERENCES MARKET_STALL(stall_id),
  source_review_id TEXT REFERENCES REVIEW(review_id),
  content TEXT,
  status TEXT DEFAULT '접수' CHECK(status IN ('접수','처리중','완료')),
  handled_by TEXT REFERENCES ADMIN(admin_id),
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS INQUIRY (
  inquiry_id TEXT PRIMARY KEY,
  user_id TEXT REFERENCES USER(user_id),
  category TEXT,
  question TEXT,
  answer TEXT,
  answered_by TEXT REFERENCES ADMIN(admin_id),
  status TEXT DEFAULT '접수' CHECK(status IN ('접수','답변완료')),
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS FEEDBACK (
  feedback_id TEXT PRIMARY KEY,
  user_id TEXT NOT NULL REFERENCES USER(user_id),
  target TEXT,
  content TEXT,
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_conversation_user ON CONVERSATION(user_id);
CREATE INDEX IF NOT EXISTS idx_message_conv ON MESSAGE(conversation_id);
CREATE INDEX IF NOT EXISTS idx_itinerary_user ON ITINERARY(user_id);
CREATE INDEX IF NOT EXISTS idx_itinerary_place_itinerary ON ITINERARY_PLACE(itinerary_id);
CREATE INDEX IF NOT EXISTS idx_stall_market ON MARKET_STALL(market_id);
CREATE INDEX IF NOT EXISTS idx_stall_merchant ON MARKET_STALL(merchant_id);
CREATE INDEX IF NOT EXISTS idx_qrcode_stall ON QR_CODE(stall_id);
CREATE INDEX IF NOT EXISTS idx_qrvisit_user ON QR_VISIT_LOG(user_id);
CREATE INDEX IF NOT EXISTS idx_review_user ON REVIEW(user_id);
CREATE INDEX IF NOT EXISTS idx_review_target ON REVIEW(target_type, target_id);
CREATE INDEX IF NOT EXISTS idx_market_geo ON MARKET(lat, lon);
"""


def init_schema():
    """ERD 확정본 그대로 스키마를 생성. 이미 테이블이 있으면 아무 것도 건드리지 않음."""
    conn = sqlite3.connect(DB_PATH)
    conn.executescript(SCHEMA_SQL)
    conn.commit()
    conn.close()


init_schema()  # 앱 시작 시 1회 실행 -> ERD 그대로 테이블 보장


# ============================================================
# 상가매칭 CSV -> MARKET / MARKET_STALL 자동 적재
# (동동 님이 가진 "전통시장_상가매칭_광주전남_FULL.csv" 하나만 있으면
#  이 노트북이 스스로 MARKET(시장)과 MARKET_STALL(점포) 테이블을 채웁니다)
# ============================================================
CSV_PATH = "전통시장_상가매칭_광주전남_FULL.csv"  # Colab에 업로드한 파일명과 동일해야 함


def load_data_from_csv(csv_path=CSV_PATH, force=False):
    import csv
    import os

    if not os.path.exists(csv_path):
        print(f"⚠️  '{csv_path}' 파일이 없어서 CSV 적재를 건너뜁니다. (좌측 파일 탭에 업로드했는지 확인해주세요)")
        return

    conn = get_conn()
    cur = conn.cursor()

    cur.execute("SELECT COUNT(*) FROM MARKET_STALL")
    already = cur.fetchone()[0]
    if already > 0 and not force:
        print(f"ℹ️  MARKET_STALL에 이미 {already}건이 있어서 CSV 적재를 건너뜁니다. (다시 넣으려면 force=True)")
        conn.close()
        return

    with open(csv_path, encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    print(f"CSV 로드 완료: {len(rows)}건")

    # 1) 시장명 기준으로 그룹핑해서 MARKET 테이블 채우기
    markets = {}
    for row in rows:
        name = row["시장명"]
        if name not in markets:
            markets[name] = {
                "시장유형": row["시장유형"],
                "시장개설주기": row["시장개설주기"],
                "취급품목": row["취급품목"],
                "지역": row["지역"],
                "lats": [],
                "lons": [],
                "addr_candidates": [],  # (거리, 도로명주소)
                "store_count": 0,
            }
        m = markets[name]
        m["store_count"] += 1
        try:
            m["lats"].append(float(row["위도"]))
            m["lons"].append(float(row["경도"]))
        except (ValueError, KeyError):
            pass
        try:
            dist = float(row["거리(m)"])
            m["addr_candidates"].append((dist, row["도로명주소"]))
        except (ValueError, KeyError):
            pass

    name_to_market_id = {}
    for name, m in markets.items():
        market_id = str(uuid.uuid4())
        name_to_market_id[name] = market_id
        lat = sum(m["lats"]) / len(m["lats"]) if m["lats"] else None
        lon = sum(m["lons"]) / len(m["lons"]) if m["lons"] else None
        # 가장 가까운 점포의 도로명주소를 시장 대표주소로 근사
        addr = min(m["addr_candidates"])[1] if m["addr_candidates"] else None

        cur.execute(
            """INSERT INTO MARKET (market_id, market_name, market_type, open_cycle,
                                    address_road, lat, lon, store_count, item_categories, data_ref_date)
               VALUES (?,?,?,?,?,?,?,?,?,?)""",
            (market_id, name, m["시장유형"], m["시장개설주기"], addr, lat, lon,
             m["store_count"], m["취급품목"], "202603"),
        )

    print(f"MARKET 테이블에 {len(markets)}개 시장 삽입 완료")

    # 2) 점포(상가) 하나하나를 MARKET_STALL에 삽입
    inserted = 0
    for row in rows:
        market_id = name_to_market_id.get(row["시장명"])
        if not market_id:
            continue
        cur.execute(
            """INSERT INTO MARKET_STALL
               (stall_id, market_id, stall_type, biz_name, category_large, category_mid,
                category_small, road_address, distance_m, source_biz_id)
               VALUES (?,?,?,?,?,?,?,?,?,?)""",
            (str(uuid.uuid4()), market_id, row["시장유형"], row["상호명"],
             row["업종대분류"], row["업종중분류"], row["업종소분류"],
             row["도로명주소"],
             float(row["거리(m)"]) if row["거리(m)"] else None,
             row["상가업소번호"]),
        )
        inserted += 1

    conn.commit()
    conn.close()
    print(f"MARKET_STALL 테이블에 {inserted}건 삽입 완료 ✅")


# ------------------------------------------------------------
# DB 연결 유틸
# ------------------------------------------------------------
def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def hash_pw(pw: str) -> str:
    # 데모용 간단 해시. 실서비스에서는 bcrypt/argon2 사용 권장.
    return hashlib.sha256(pw.encode("utf-8")).hexdigest()


load_data_from_csv()  # get_conn 정의 이후에 실행 (이미 데이터 있으면 자동으로 건너뜀)


def ensure_user_extra_columns():
    """USER 테이블에 온보딩용 컬럼(선호 여행 스타일)이 없으면 추가 (FR-007 대응)."""
    conn = get_conn()
    cur = conn.cursor()
    try:
        cur.execute("ALTER TABLE USER ADD COLUMN travel_style TEXT")
    except sqlite3.OperationalError:
        pass  # 이미 컬럼이 있으면 무시
    conn.commit()
    conn.close()


ensure_user_extra_columns()

# FR-007 온보딩(추가 프로필 입력)의 "선호 여행 스타일(다중선택)" 항목
# PRD에 구체 목록이 없어 전통시장·지역관광 앱 성격에 맞춰 7개로 정의함 (필요하면 자유롭게 수정하세요)
TRAVEL_STYLE_OPTIONS = ["자연/힐링", "맛집/미식", "전통시장", "문화/역사", "액티비티/체험", "카페/디저트", "사진/인생샷"]


# ------------------------------------------------------------
# 1) 회원가입 / 로그인 / 로그아웃
# ------------------------------------------------------------
def signup(auth_provider, name, email, birthdate, password, password_confirm, travel_styles):
    """
    auth_provider: 'kakao' | 'naver' | 'google' | 'email'
    ※ 프로토타입 한계: 실제 카카오/네이버/구글 OAuth 연동은 Colab 환경에서
      리다이렉트 처리가 되지 않아 여기서는 '이메일 입력만으로 해당 제공자로 가입했다'고
      시뮬레이션합니다. 실서비스에서는 이 부분을 각 사 SDK 연동으로 교체하면 됩니다.
    """
    if not name or not email:
        return "❌ 이름과 이메일은 필수입니다."

    if auth_provider == "email":
        if not password:
            return "❌ 비밀번호를 입력해주세요."
        if len(password) < 8:
            return "❌ 비밀번호는 8자 이상이어야 합니다."
        if password != password_confirm:
            return "❌ 비밀번호와 비밀번호 확인이 일치하지 않습니다."

    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT user_id, auth_provider FROM USER WHERE email = ?", (email,))
    existing = cur.fetchone()
    if existing:
        conn.close()
        if existing[1] != auth_provider:
            return f"❌ 이미 '{existing[1]}'(으)로 가입된 이메일입니다. 해당 방법으로 로그인해주세요."
        return "❌ 이미 가입된 이메일입니다."

    user_id = str(uuid.uuid4())
    style_str = ",".join(travel_styles) if travel_styles else None
    pw_field = hash_pw(password) if auth_provider == "email" else None  # 소셜 로그인은 비밀번호 없음

    cur.execute(
        """INSERT INTO USER (user_id, name, email, birthdate, auth_provider,
                              auth_provider_id, identity_verified, status, created_at, travel_style)
           VALUES (?, ?, ?, ?, ?, ?, 0, 'active', ?, ?)""",
        (user_id, name, email, birthdate, auth_provider, pw_field, datetime.now().isoformat(), style_str),
    )
    conn.commit()
    conn.close()

    provider_label = {"kakao": "카카오", "naver": "네이버", "google": "구글", "email": "이메일(자체)"}[auth_provider]
    return f"✅ {provider_label} 회원가입 완료! (user_id: {user_id[:8]}...) 이제 로그인 탭에서 로그인해주세요."


def login(auth_provider, email, password):
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(
        "SELECT user_id, name, auth_provider, auth_provider_id FROM USER WHERE email = ?", (email,)
    )
    row = cur.fetchone()
    conn.close()

    if not row:
        return "❌ 등록되지 않은 이메일입니다.", None
    user_id, name, real_provider, pw_hash = row

    if real_provider != auth_provider:
        provider_label = {"kakao": "카카오", "naver": "네이버", "google": "구글", "email": "이메일(자체)"}[real_provider]
        return f"❌ 이 계정은 '{provider_label}'로 가입되어 있습니다. 해당 방법으로 로그인해주세요.", None

    if auth_provider == "email":
        if pw_hash != hash_pw(password):
            return "❌ 아이디 또는 비밀번호를 확인하세요.", None
    # 소셜 로그인(카카오/네이버/구글)은 프로토타입에서 이메일 일치만 확인 (실제로는 OAuth 토큰 검증)

    return f"✅ {name}님 환영합니다!", user_id


def logout():
    return "👋 로그아웃되었습니다.", None



# ------------------------------------------------------------
# 2) 지역 검색 (전통시장 둘러보기)
# ------------------------------------------------------------
def get_region_list():
    conn = get_conn()
    cur = conn.cursor()
    # address_road 첫 토큰(시도) 기준으로 지역 추출
    cur.execute("SELECT DISTINCT address_road FROM MARKET")
    rows = cur.fetchall()
    conn.close()
    regions = sorted({r[0].split()[0] for r in rows if r[0]})
    return ["전체"] + regions


def search_markets(region, market_type, keyword):
    conn = get_conn()
    cur = conn.cursor()
    query = "SELECT market_name, market_type, open_cycle, address_road, store_count, item_categories FROM MARKET WHERE 1=1"
    params = []
    if region and region != "전체":
        query += " AND address_road LIKE ?"
        params.append(f"{region}%")
    if market_type and market_type != "전체":
        query += " AND market_type LIKE ?"
        params.append(f"%{market_type}%")
    if keyword:
        query += " AND market_name LIKE ?"
        params.append(f"%{keyword}%")
    query += " ORDER BY store_count DESC"
    cur.execute(query, params)
    rows = cur.fetchall()
    conn.close()

    if not rows:
        return [["검색 결과가 없습니다.", "", "", "", "", ""]]
    return [list(r) for r in rows]


def get_market_type_list():
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT DISTINCT market_type FROM MARKET")
    rows = cur.fetchall()
    conn.close()
    return ["전체"] + sorted({r[0] for r in rows if r[0]})


def get_market_name_list():
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT market_name FROM MARKET ORDER BY market_name")
    rows = cur.fetchall()
    conn.close()
    return [r[0] for r in rows]


# ------------------------------------------------------------
# 3) 시장 상세 (내부 점포 목록)
# ------------------------------------------------------------
def get_market_detail(market_name):
    if not market_name:
        return "시장을 선택해주세요.", [], []

    conn = get_conn()
    cur = conn.cursor()
    cur.execute(
        """SELECT market_name, market_type, open_cycle, address_road,
                  store_count, item_categories, phone
           FROM MARKET WHERE market_name = ?""",
        (market_name,),
    )
    market = cur.fetchone()
    if not market:
        conn.close()
        return "시장 정보를 찾을 수 없습니다.", [], []

    info_md = f"""### {market[0]}
- **유형**: {market[1]}
- **개설주기**: {market[2] or '상설(매일)'}
- **주소**: {market[3]}
- **공식 점포수**: {market[4]}
- **취급품목**: {market[5]}
- **전화번호**: {market[6] or '정보없음'}
"""

    cur.execute("SELECT DISTINCT category_large FROM MARKET_STALL WHERE market_id = (SELECT market_id FROM MARKET WHERE market_name = ?)", (market_name,))
    categories = ["전체"] + sorted({r[0] for r in cur.fetchall() if r[0]})

    cur.execute(
        """SELECT biz_name, category_large, category_small, road_address, distance_m
           FROM MARKET_STALL
           WHERE market_id = (SELECT market_id FROM MARKET WHERE market_name = ?)
           ORDER BY distance_m ASC LIMIT 200""",
        (market_name,),
    )
    stalls = cur.fetchall()
    conn.close()

    return info_md, categories, [list(s) for s in stalls]


def filter_market_stalls(market_name, category):
    if not market_name:
        return []
    conn = get_conn()
    cur = conn.cursor()
    query = """SELECT biz_name, category_large, category_small, road_address, distance_m
               FROM MARKET_STALL
               WHERE market_id = (SELECT market_id FROM MARKET WHERE market_name = ?)"""
    params = [market_name]
    if category and category != "전체":
        query += " AND category_large = ?"
        params.append(category)
    query += " ORDER BY distance_m ASC LIMIT 200"
    cur.execute(query, params)
    rows = cur.fetchall()
    conn.close()
    return [list(r) for r in rows]


# ------------------------------------------------------------
# 4) 찜하기 (즐겨찾기)
# ------------------------------------------------------------
def add_favorite(user_id, market_name):
    if not user_id:
        return "❌ 먼저 로그인해주세요."
    if not market_name:
        return "❌ 찜할 시장을 선택해주세요."

    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT market_id FROM MARKET WHERE market_name = ?", (market_name,))
    m = cur.fetchone()
    if not m:
        conn.close()
        return "❌ 시장 정보를 찾을 수 없습니다."
    market_id = m[0]

    cur.execute(
        "SELECT favorite_id FROM FAVORITE WHERE user_id=? AND target_type='market' AND target_id=?",
        (user_id, market_id),
    )
    if cur.fetchone():
        conn.close()
        return "이미 찜한 시장입니다."

    cur.execute(
        "INSERT INTO FAVORITE (favorite_id, user_id, target_type, target_id, created_at) VALUES (?,?,?,?,?)",
        (str(uuid.uuid4()), user_id, "market", market_id, datetime.now().isoformat()),
    )
    conn.commit()
    conn.close()
    return f"✅ '{market_name}'을(를) 찜 목록에 추가했습니다."


# ------------------------------------------------------------
# 5) 챗봇 (대화 세션 + 메시지 기록)
# ------------------------------------------------------------
def generate_ai_response(user_message, conn):
    """
    데모용 간단 응답 로직 (실제 서비스에서는 이 함수 내부만
    Claude API 호출로 교체하면 됩니다 - 조건 슬롯 추출 후 MARKET/PLACE 검색 결과를 근거로 답변).
    지금은 사용자가 입력한 키워드로 MARKET을 간단 검색해서 결과를 붙여주는 수준.
    """
    cur = conn.cursor()
    cur.execute(
        "SELECT market_name, market_type, address_road FROM MARKET WHERE market_name LIKE ? LIMIT 3",
        (f"%{user_message.strip()}%",),
    )
    hits = cur.fetchall()
    if hits:
        lines = [f"- {h[0]} ({h[1]}, {h[2]})" for h in hits]
        return "말씀하신 내용과 관련된 전통시장을 찾아봤어요:\n" + "\n".join(lines)
    return f"'{user_message}' 관련해서 찾아보고 있어요. (이 자리에 실제 추천 로직/Claude API 응답이 들어갈 자리입니다)"


def get_user_conversations(user_id):
    if not user_id:
        return []
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(
        "SELECT conversation_id, title FROM CONVERSATION WHERE user_id=? AND is_deleted=0 ORDER BY updated_at DESC",
        (user_id,),
    )
    rows = cur.fetchall()
    conn.close()
    return [f"{r[1] or '(제목없음)'} | {r[0]}" for r in rows]


def start_new_conversation(user_id):
    if not user_id:
        return "❌ 먼저 로그인해주세요.", gr.update(choices=[]), []

    conn = get_conn()
    cur = conn.cursor()
    conv_id = str(uuid.uuid4())
    now = datetime.now().isoformat()
    cur.execute(
        "INSERT INTO CONVERSATION (conversation_id, user_id, title, is_deleted, created_at, updated_at) VALUES (?,?,?,0,?,?)",
        (conv_id, user_id, "새 대화", now, now),
    )
    conn.commit()
    conn.close()

    choices = get_user_conversations(user_id)
    selected = f"새 대화 | {conv_id}"
    return f"✅ 새 대화를 시작했습니다.", gr.update(choices=choices, value=selected), []


def load_conversation(conv_choice):
    if not conv_choice:
        return []
    conv_id = conv_choice.split("|")[-1].strip()
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(
        "SELECT sender, content FROM MESSAGE WHERE conversation_id=? ORDER BY created_at ASC",
        (conv_id,),
    )
    rows = cur.fetchall()
    conn.close()

    history = []
    for sender, content in rows:
        if sender == "user":
            history.append([content, None])
        else:
            if history and history[-1][1] is None:
                history[-1][1] = content
            else:
                history.append([None, content])
    return history


def send_message(user_id, conv_choice, message, history):
    if not user_id:
        return history, "", gr.update()
    if not conv_choice:
        return history, message, gr.update()
    if not message or not message.strip():
        return history, "", gr.update()

    conv_id = conv_choice.split("|")[-1].strip()
    conn = get_conn()
    cur = conn.cursor()
    now = datetime.now().isoformat()

    # 1) 사용자 메시지 기록
    cur.execute(
        "INSERT INTO MESSAGE (message_id, conversation_id, sender, content, created_at) VALUES (?,?,?,?,?)",
        (str(uuid.uuid4()), conv_id, "user", message, now),
    )

    # 2) AI 응답 생성 + 기록
    ai_reply = generate_ai_response(message, conn)
    cur.execute(
        "INSERT INTO MESSAGE (message_id, conversation_id, sender, content, created_at) VALUES (?,?,?,?,?)",
        (str(uuid.uuid4()), conv_id, "ai", ai_reply, datetime.now().isoformat()),
    )

    # 3) 대화 제목이 "새 대화"면 첫 사용자 메시지로 제목 갱신 + updated_at 갱신
    cur.execute("SELECT title FROM CONVERSATION WHERE conversation_id=?", (conv_id,))
    row = cur.fetchone()
    if row and row[0] == "새 대화":
        cur.execute(
            "UPDATE CONVERSATION SET title=?, updated_at=? WHERE conversation_id=?",
            (message[:20], datetime.now().isoformat(), conv_id),
        )
    else:
        cur.execute(
            "UPDATE CONVERSATION SET updated_at=? WHERE conversation_id=?",
            (datetime.now().isoformat(), conv_id),
        )

    conn.commit()
    conn.close()

    history = history + [[message, ai_reply]]
    return history, "", gr.update(choices=get_user_conversations(user_id))


def get_my_favorites(user_id):
    if not user_id:
        return [["로그인이 필요합니다.", "", ""]]
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(
        """SELECT m.market_name, m.market_type, m.address_road
           FROM FAVORITE f JOIN MARKET m ON f.target_id = m.market_id
           WHERE f.user_id = ? AND f.target_type = 'market'
           ORDER BY f.created_at DESC""",
        (user_id,),
    )
    rows = cur.fetchall()
    conn.close()
    if not rows:
        return [["찜한 시장이 없습니다.", "", ""]]
    return [list(r) for r in rows]


# ------------------------------------------------------------
# Gradio UI
# ------------------------------------------------------------
with gr.Blocks(title="AI 전통시장 여행 플래너") as demo:
    gr.Markdown("# 🏮 AI기반 전통시장 여행 플래너 (프로토타입)")

    user_state = gr.State(None)  # 로그인한 user_id 저장

    with gr.Tab("회원가입"):
        gr.Markdown("### 회원가입")
        gr.Markdown(
            "_프로토타입 안내: 카카오/네이버/구글은 실제 OAuth 연동 대신, "
            "이메일 입력만으로 해당 방식으로 가입한 것처럼 시뮬레이션합니다._"
        )
        su_provider = gr.Radio(
            choices=[("카카오", "kakao"), ("네이버", "naver"), ("구글", "google"), ("자체(이메일)", "email")],
            value="email",
            label="가입 방식",
        )
        su_name = gr.Textbox(label="이름")
        su_email = gr.Textbox(label="이메일")
        su_birth = gr.Textbox(label="생년월일 (YYYY-MM-DD)")

        with gr.Group(visible=True) as su_pw_group:
            su_pw = gr.Textbox(label="비밀번호 (8자 이상)", type="password")
            su_pw_confirm = gr.Textbox(label="비밀번호 확인", type="password")

        su_travel_style = gr.CheckboxGroup(
            choices=TRAVEL_STYLE_OPTIONS,
            label="선호하는 여행 스타일 (다중 선택)",
        )
        su_btn = gr.Button("회원가입", variant="primary")
        su_result = gr.Markdown()

        def _toggle_pw_group(provider):
            return gr.update(visible=(provider == "email"))

        su_provider.change(_toggle_pw_group, su_provider, su_pw_group)

        su_btn.click(
            signup,
            [su_provider, su_name, su_email, su_birth, su_pw, su_pw_confirm, su_travel_style],
            su_result,
        )

    with gr.Tab("로그인"):
        gr.Markdown("### 로그인")
        li_provider = gr.Radio(
            choices=[("카카오", "kakao"), ("네이버", "naver"), ("구글", "google"), ("자체(이메일)", "email")],
            value="email",
            label="로그인 방식",
        )
        li_email = gr.Textbox(label="이메일")
        li_pw = gr.Textbox(label="비밀번호 (자체 로그인만 해당)", type="password")
        with gr.Row():
            li_btn = gr.Button("로그인", variant="primary")
            lo_btn = gr.Button("로그아웃")
        li_result = gr.Markdown()
        # (로그인 버튼 이벤트는 이 파일 아래쪽에서 챗봇 탭의 대화 목록 갱신과 함께 연결됩니다)
        lo_btn.click(logout, None, [li_result, user_state])

    with gr.Tab("지역 검색 (전통시장 둘러보기)"):
        gr.Markdown("### 조건으로 전통시장 찾기")
        with gr.Row():
            region_dd = gr.Dropdown(choices=get_region_list(), value="전체", label="지역(시/도)")
            type_dd = gr.Dropdown(choices=get_market_type_list(), value="전체", label="시장 유형")
            keyword_tb = gr.Textbox(label="시장명 검색")
        search_btn = gr.Button("검색", variant="primary")
        result_table = gr.Dataframe(
            headers=["시장명", "유형", "개설주기", "주소", "점포수", "취급품목"],
            interactive=False,
        )
        search_btn.click(search_markets, [region_dd, type_dd, keyword_tb], result_table)

    with gr.Tab("시장 상세"):
        gr.Markdown("### 시장을 선택하면 내부 점포 목록을 볼 수 있어요")
        market_dd = gr.Dropdown(choices=get_market_name_list(), label="시장 선택")
        detail_md = gr.Markdown()
        category_dd = gr.Dropdown(choices=["전체"], label="업종 필터", value="전체")
        stall_table = gr.Dataframe(
            headers=["상호명", "업종대분류", "업종소분류", "도로명주소", "거리(m)"],
            interactive=False,
        )
        with gr.Row():
            fav_btn = gr.Button("❤️ 이 시장 찜하기")
        fav_result = gr.Markdown()

        market_dd.change(get_market_detail, market_dd, [detail_md, category_dd, stall_table])
        category_dd.change(filter_market_stalls, [market_dd, category_dd], stall_table)
        fav_btn.click(add_favorite, [user_state, market_dd], fav_result)

    with gr.Tab("챗봇"):
        gr.Markdown("### AI 여행 플래너 챗봇 (대화가 CONVERSATION/MESSAGE 테이블에 실제로 기록됩니다)")
        with gr.Row():
            conv_dd = gr.Dropdown(choices=[], label="내 대화 목록 (선택해서 이어가기)")
            new_conv_btn = gr.Button("➕ 새 대화 시작")
        conv_status = gr.Markdown()
        chatbot_ui = gr.Chatbot(label="대화창", height=350)
        msg_tb = gr.Textbox(label="메시지 입력 (예: 순천, 여수 같은 시장명 키워드로 검색해보세요)")
        send_btn = gr.Button("전송", variant="primary")

        new_conv_btn.click(start_new_conversation, user_state, [conv_status, conv_dd, chatbot_ui])
        conv_dd.change(load_conversation, conv_dd, chatbot_ui)
        send_btn.click(send_message, [user_state, conv_dd, msg_tb, chatbot_ui], [chatbot_ui, msg_tb, conv_dd])
        msg_tb.submit(send_message, [user_state, conv_dd, msg_tb, chatbot_ui], [chatbot_ui, msg_tb, conv_dd])

    with gr.Tab("마이페이지"):
        gr.Markdown("### 내 찜 목록")
        my_fav_btn = gr.Button("새로고침")
        my_fav_table = gr.Dataframe(
            headers=["시장명", "유형", "주소"], interactive=False
        )
        my_fav_btn.click(get_my_favorites, user_state, my_fav_table)

    # 로그인 성공 -> user_state 갱신 -> 챗봇 탭 "내 대화 목록" 자동 갱신 (순서 보장을 위해 .then() 체이닝)
    login_event = li_btn.click(login, [li_provider, li_email, li_pw], [li_result, user_state])
    login_event.then(lambda uid: gr.update(choices=get_user_conversations(uid)), user_state, conv_dd)

## 3. 앱 실행
아래 셀을 실행하면 `share=True`로 공개 링크가 생성됩니다. (Colab 세션이 꺼지면 링크도 만료돼요)

In [ ]:
demo.launch(share=True, debug=True)